# Lab 1a: Azure Cosmos DB Vector Store Integration

**Time:** ~20-25 minutes | **Prerequisites:** Azure subscription, Python 3.10+

## What You'll Learn

| Concept | Why It Matters |
|---------|---------------|
| **Vector embeddings** | Turn text into numbers AI can search |
| **Cosmos DB DiskANN** | Fast similarity search at scale |
| **RAG pattern** | Give your agent knowledge without retraining |

---

## Step 1: Environment Setup

Load environment variables and import required packages.

In [ ]:
import os
import json
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify required variables
required_vars = ["COSMOS_ENDPOINT", "AZURE_AI_PROJECT_ENDPOINT"]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    print(f"Missing: {missing}")
    print("Tip: Run ./scripts/setup-cosmosdb.sh to provision resources")
else:
    print("Environment ready!")

## Step 2: Connect to Cosmos DB

Create a client using Azure Identity (passwordless auth).

In [ ]:
from azure.cosmos import CosmosClient
from azure.identity import DefaultAzureCredential

# Connect with managed identity / Azure CLI credentials
credential = DefaultAzureCredential()
client = CosmosClient(os.getenv("COSMOS_ENDPOINT"), credential)

# Get database and container
database = client.get_database_client(os.getenv("COSMOS_DATABASE", "knowledge-db"))
container = database.get_container_client(os.getenv("COSMOS_CONTAINER", "documents"))

# Quick test
print(f"Connected to: {database.id}/{container.id}")
print(f"Document count: {len(list(container.query_items('SELECT c.id FROM c', enable_cross_partition_query=True)))}")

## Step 3: Understanding Vector Embeddings

**Key Insight:** Embeddings convert text to vectors where similar meanings are close together.

Let's see this in action:

In [ ]:
from openai import AzureOpenAI
from azure.identity import get_bearer_token_provider
import numpy as np

# Create a proper token provider for Azure AI Foundry authentication
token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default"
)

# Create embedding client (uses Azure AI Foundry endpoint)
embedding_client = AzureOpenAI(
    azure_endpoint="https://aifoundry825233136833-resource.cognitiveservices.azure.com/",
    # azure_endpoint=os.getenv("PROJECT_ENDPOINT"),
    azure_ad_token_provider=token_provider,
    api_version="2024-02-01"
)

def get_embedding(text: str) -> list[float]:
    """Convert text to a 1536-dimensional vector."""
    response = embedding_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Demo: Similar concepts have high similarity scores
queries = [
    "database is slow",
    "SQL query performance issues",
    "the weather is nice today"
]

embeddings = [get_embedding(q) for q in queries]

print("Similarity Matrix:")
print(f"'{queries[0]}' vs '{queries[1]}': {cosine_similarity(embeddings[0], embeddings[1]):.3f} (related!)")
print(f"'{queries[0]}' vs '{queries[2]}': {cosine_similarity(embeddings[0], embeddings[2]):.3f} (unrelated)")

## Step 4: Vector Search in Cosmos DB

Now let's search our knowledge base using vector similarity.

**The Magic:** DiskANN finds similar documents in milliseconds, even with millions of vectors.

In [ ]:
def vector_search(query: str, top_k: int = 3) -> list[dict]:
    """
    Search knowledge base using vector similarity.
    
    This is the core of RAG - finding relevant documents
    to give context to the AI.
    """
    # Convert query to vector
    query_vector = get_embedding(query)
    
    # Cosmos DB vector search query
    results = container.query_items(
        query="""
            SELECT TOP @top_k 
                c.id, c.title, c.content, c.category,
                VectorDistance(c.contentVector, @queryVector) as similarity
            FROM c
            ORDER BY VectorDistance(c.contentVector, @queryVector)
        """,
        parameters=[
            {"name": "@top_k", "value": top_k},
            {"name": "@queryVector", "value": query_vector}
        ],
        enable_cross_partition_query=True
    )
    
    return list(results)

# Test vector search
print("Searching: 'database running slow'\n")
results = vector_search("database running slow")

for doc in results:
    print(f"📄 {doc['title']}")
    print(f"   Category: {doc['category']}")
    print(f"   Similarity: {1 - doc['similarity']:.3f}")  # Convert distance to similarity
    print()

## Step 5: Create RAG Tool for Agent

Now we wrap our vector search as a tool the Agent Framework can use.

In [ ]:
from agent_framework import tool

@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the SRE knowledge base for relevant documentation.
    
    Use this tool when you need to find:
    - Runbooks for troubleshooting
    - Playbooks for incident response  
    - Best practices for operations
    
    Args:
        query: Natural language description of what you're looking for
    
    Returns:
        Relevant documentation excerpts with source references
    """
    results = vector_search(query, top_k=3)
    
    if not results:
        return "No relevant documentation found."
    
    # Format results for the agent
    output = []
    for doc in results:
        output.append(f"### {doc['title']}\n")
        output.append(f"Category: {doc['category']}\n")
        output.append(f"{doc['content'][:500]}...\n")
        output.append(f"Source: {doc['id']}\n")
        output.append("---")
    
    return "\n".join(output)

print("Tool created:", search_knowledge_base.name)
print("Description:", search_knowledge_base.description[:100] + "...")

## Step 6: Build the RAG-Enabled Agent

Create an agent that uses our knowledge base tool.

In [ ]:
from agent_framework import Agent
from agent_framework.azure import AzureAIAgentClient

# Create the model
client = AzureAIAgentClient(
    project_endpoint=os.getenv("PROJECT_ENDPOINT"),
    credential=credential,
    model_deployment_name="gpt-5"
)


# Create agent with RAG tool
sre_agent = Agent(
    name="SRE Assistant",
    client=client,
    instructions="""
    You are an SRE assistant with access to the team's knowledge base.
    
    When answering questions:
    1. ALWAYS search the knowledge base first for relevant docs
    2. Base your answers on the retrieved documentation
    3. Cite your sources by document title
    4. If no relevant docs found, say so clearly
    """,
    tools=[search_knowledge_base]
)

print(f"Agent '{sre_agent.name}' ready with {len(sre_agent.default_options['tools'])} tool(s)")

## Step 7: Test the Agent

Ask questions and watch the agent use RAG!

In [ ]:
# Test question about database performance
response = await sre_agent.run(
    "Our production database is running slow. What should I check first?"
)

print("Agent Response:")
print(response)

In [ ]:
# Test question about incident response
response = await sre_agent.run(
    "We have a SEV1 incident. What's the protocol?"
)

print("Agent Response:")
print(response.content)

## Step 8: Compare with Cosmos DB Data Agent (Optional)

Azure Cosmos DB has a built-in "Data Agent" feature. Let's compare approaches:

| Aspect | Our RAG Agent | Cosmos DB Data Agent |
|--------|---------------|---------------------|
| **Setup** | Full control, code-first | Zero-code, portal UI |
| **Customization** | Unlimited | Limited to built-in options |
| **Embedding Model** | Your choice | Built-in models |
| **Integration** | Any agent framework | Cosmos DB ecosystem |
| **Cost** | Pay for what you use | Premium tier feature |

**When to use which:**
- **Our approach:** Full control, multi-source RAG, custom agents
- **Data Agent:** Quick prototypes, Cosmos-only scenarios

In [ ]:
# To try Cosmos DB Data Agent:
# 1. Go to Azure Portal > your Cosmos DB account
# 2. Click "Data Agent" in the left menu  
# 3. Follow the setup wizard
# 4. Ask it the same questions we asked our agent

print("Try asking the Data Agent:")
print("  - 'What documents do I have about databases?'")
print("  - 'Summarize the SEV1 response playbook'")
print("\nCompare the responses with our custom agent!")

---

## Summary

You learned:

1. **Vector embeddings** turn text into searchable numbers
2. **Cosmos DB DiskANN** enables fast similarity search
3. **RAG pattern** = retrieve docs + augment prompt + generate
4. **Tool wrapping** makes any function agent-callable

**Next Steps:**
- Add more documents to the knowledge base
- Experiment with different embedding models
- Try hybrid search (vector + keyword filters)

---

*Lab complete! Time: ~20-25 minutes*

# Lab 1a: Azure Cosmos DB Vector Store Integration

**Duration:** 20-25 minutes  
**Objective:** Connect your SRE agent to a vector store for semantic search over operational knowledge

---

## What You'll Learn

| Concept | What It Is | Why It Matters |
|---------|------------|----------------|
| **Vector Store** | Database that stores text as numbers (embeddings) | Enables "find similar content" instead of exact keyword match |
| **Embeddings** | AI-generated numerical representation of text | "Database performance issue" ≈ "DB slowdown" (semantically similar) |
| **RAG** | Retrieval Augmented Generation | Agent retrieves relevant docs before answering → better, grounded responses |

### The "Aha!" Moment

Ask the agent *"How do I fix a slow database?"* → It searches your knowledge base, finds the right runbook, and gives you **step-by-step instructions from YOUR documentation**.

---

## Prerequisites

- ✅ Lab 1 completed (Agent basics)
- ✅ Azure subscription
- ✅ Azure AI Foundry project with models deployed
- ✅ Cosmos DB set up (run `./scripts/setup-cosmosdb.sh` if not done)
- ✅ Knowledge base loaded (run `python scripts/load_knowledge_base.py` if not done)

---

## Step 1: Environment Setup

Install dependencies and configure connections to Azure AI Foundry and Cosmos DB.

In [ ]:
# Install dependencies
%pip install agent-framework --pre azure-cosmos openai python-dotenv --quiet

In [ ]:
import os
from typing import Annotated
from pydantic import Field
from dotenv import load_dotenv

from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.cosmos import CosmosClient
from openai import AzureOpenAI

# Load environment variables
load_dotenv()

# Configuration
PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
COSMOS_ENDPOINT = os.getenv("COSMOS_ENDPOINT")
COSMOS_DATABASE = os.getenv("COSMOS_DATABASE", "sre-data")
COSMOS_CONTAINER = os.getenv("COSMOS_CONTAINER", "sre-knowledge")

# Validate configuration
if not PROJECT_ENDPOINT or PROJECT_ENDPOINT.startswith("https://<"):
    raise ValueError("Please set PROJECT_ENDPOINT in your .env file")
if not COSMOS_ENDPOINT or COSMOS_ENDPOINT.startswith("https://<"):
    raise ValueError("Please set COSMOS_ENDPOINT in your .env file (run ./scripts/setup-cosmosdb.sh)")

# Initialize credential
credential = DefaultAzureCredential()

print("✓ Environment loaded")
print(f"  Project endpoint: {PROJECT_ENDPOINT[:50]}...")
print(f"  Cosmos endpoint: {COSMOS_ENDPOINT[:50]}...")

---

## Step 2: Connect to Cosmos DB

Initialize the Cosmos DB client to access the knowledge base container.

In [ ]:
# Connect to Cosmos DB
cosmos_client = CosmosClient(url=COSMOS_ENDPOINT, credential=credential)
database = cosmos_client.get_database_client(COSMOS_DATABASE)
container = database.get_container_client(COSMOS_CONTAINER)

# Verify connection - list documents
docs = list(container.query_items(
    "SELECT c.id, c.category, c.title FROM c",
    enable_cross_partition_query=True
))

print(f"✓ Connected to Cosmos DB")
print(f"\nDocuments in knowledge base ({len(docs)} total):")
for doc in docs:
    print(f"  [{doc['category']}] {doc['title']}")

---

## Step 3: Set Up Embeddings Client

We need an embedding model to convert search queries into vectors (numbers) that can be compared with our stored document vectors.

In [ ]:
# Initialize embedding client
token_provider = get_bearer_token_provider(
    credential, 
    "https://cognitiveservices.azure.com/.default"
)

embedding_client = AzureOpenAI(
    azure_endpoint=PROJECT_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2024-10-21",
)

def generate_embedding(text: str) -> list[float]:
    """Convert text to a vector (1536 numbers) using Azure OpenAI."""
    response = embedding_client.embeddings.create(
        model="text-embedding-3-small",
        input=text,
        dimensions=1536
    )
    return response.data[0].embedding

# Test embedding generation
test_embedding = generate_embedding("database performance")
print(f"✓ Embedding client ready")
print(f"  Test embedding: [{test_embedding[0]:.4f}, {test_embedding[1]:.4f}, ...] ({len(test_embedding)} dimensions)")

---

## Step 4: Create the Vector Search Tool

This is the **key function** that enables RAG. It:
1. Converts your question into a vector (embedding)
2. Finds the most similar documents in Cosmos DB
3. Returns the relevant content for the agent to use

> **Semantic search understands meaning, not just keywords.**  
> "slow database" will match documents about "performance degradation" even if those exact words aren't used.

In [ ]:
def search_knowledge_base(
    query: Annotated[str, Field(description="Natural language search query, e.g. 'how to fix slow database'")]
) -> list[dict]:
    """
    Search the SRE knowledge base for relevant runbooks, playbooks, and guides.
    Uses semantic similarity to find documents that match the meaning of your query.
    """
    # Step 1: Convert query to embedding
    query_embedding = generate_embedding(query)
    
    # Step 2: Vector search in Cosmos DB
    # VectorDistance finds documents with similar embeddings
    results = list(container.query_items(
        query="""
            SELECT TOP 3 
                c.id, 
                c.title, 
                c.category,
                c.content,
                VectorDistance(c.contentVector, @embedding) AS similarity
            FROM c
            ORDER BY VectorDistance(c.contentVector, @embedding)
        """,
        parameters=[{"name": "@embedding", "value": query_embedding}],
        enable_cross_partition_query=True
    ))
    
    # Return simplified results for the agent
    return [
        {
            "title": r["title"],
            "category": r["category"],
            "content": r["content"][:2000],  # Truncate for context window
            "similarity": round(r["similarity"], 4)
        }
        for r in results
    ]

print("✓ Tool defined: search_knowledge_base")

### Test the Vector Search

Let's test the search tool directly to see how it works.

In [ ]:
# Test: Search for database-related content
print("Query: 'slow database performance'")
print("=" * 50)

results = search_knowledge_base("slow database performance")

for i, doc in enumerate(results, 1):
    print(f"\n{i}. {doc['title']} ({doc['category']})")
    print(f"   Similarity: {doc['similarity']}")
    print(f"   Preview: {doc['content'][:200]}...")

In [ ]:
# Test: Different query, same meaning
# This demonstrates semantic search - finding related content without exact keyword match
print("Query: 'db is running slowly' (different words, similar meaning)")
print("=" * 50)

results = search_knowledge_base("db is running slowly")

for i, doc in enumerate(results, 1):
    print(f"\n{i}. {doc['title']} ({doc['category']})")
    print(f"   Similarity: {doc['similarity']}")

---

## Step 5: Create the RAG-Enabled Agent

Now we'll create an agent that uses the `search_knowledge_base` tool to ground its responses in your actual documentation.

In [ ]:
# Re-use the metrics tool from Lab 1
def get_system_metrics(
    server_name: Annotated[str, Field(description="Server name (e.g., vm-prod-01, vm-db-01)")],
    metric_type: Annotated[str, Field(description="Metric type: cpu, memory, disk, network_in, or all")] = "all"
) -> dict:
    """Get current system metrics for a server."""
    metrics_data = {
        "vm-prod-01": {"cpu": 78.5, "memory": 62.3, "disk": 45.0, "network_in": 125.6},
        "vm-prod-02": {"cpu": 23.1, "memory": 41.2, "disk": 72.8, "network_in": 89.3},
        "vm-db-01": {"cpu": 91.2, "memory": 88.5, "disk": 55.0, "network_in": 234.1},
    }
    server = metrics_data.get(server_name, {})
    if not server:
        return {"error": f"Server {server_name} not found"}
    if metric_type == "all":
        return {"server": server_name, "metrics": server}
    return {"server": server_name, "metric": metric_type, "value": server.get(metric_type)}

print("✓ Tool defined: get_system_metrics")

In [ ]:
# Create the RAG-enabled agent
chat_client = AzureAIAgentClient(
    project_endpoint=PROJECT_ENDPOINT,
    credential=credential,
    model_deployment_name="gpt-4o"
)

sre_agent = ChatAgent(
    chat_client=chat_client,
    name="SRE-Assistant-RAG",
    instructions="""
You are an expert Site Reliability Engineer assistant with access to the organization's 
knowledge base of runbooks, playbooks, and troubleshooting guides.

IMPORTANT: When diagnosing issues or answering questions:
1. ALWAYS search the knowledge base first using search_knowledge_base
2. Use get_system_metrics to gather current data when relevant
3. Base your recommendations on the documented procedures
4. Cite which document you're referencing (e.g., "According to the Database Performance Runbook...")

Be specific and actionable. Use the actual steps from the runbooks.
""",
    tools=[search_knowledge_base, get_system_metrics],
)

print(f"✓ Agent created: {sre_agent.name}")
print(f"  Tools: [search_knowledge_base, get_system_metrics]")

---

## Step 6: Test RAG in Action

Now for the magic! Ask the agent about database issues and watch it:
1. Search the knowledge base
2. Find the relevant runbook
3. Provide specific, documented procedures

In [ ]:
# Create a conversation thread
thread = sre_agent.get_new_thread()

# Ask about database performance
query = "vm-db-01 is showing high CPU. What should I do?"

print(f"User: {query}")
print("\n" + "=" * 60)
print("Agent is searching knowledge base and analyzing...")
print("=" * 60 + "\n")

response = await sre_agent.run(query, thread=thread)

print("AGENT RESPONSE:")
print("-" * 60)
print(response.text)

In [ ]:
# Follow-up question - the agent remembers context
follow_up = "What if the CPU stays high after killing the queries?"

print(f"User: {follow_up}")
print("\n" + "=" * 60)

response2 = await sre_agent.run(follow_up, thread=thread)

print("AGENT RESPONSE:")
print("-" * 60)
print(response2.text)

---

## Step 7: Compare - With vs Without RAG

Let's create an agent WITHOUT the knowledge base tool to see the difference.

In [ ]:
# Agent without RAG (no knowledge base access)
basic_agent = ChatAgent(
    chat_client=chat_client,
    name="SRE-Assistant-Basic",
    instructions="""You are an SRE assistant. Help diagnose infrastructure issues.
Be concise and actionable.""",
    tools=[get_system_metrics],  # Only metrics, no knowledge base
)

print("Created basic agent (without knowledge base)")

In [ ]:
# Ask the same question to both agents
test_query = "How do I troubleshoot high CPU on a database server?"

print("QUESTION:", test_query)
print("\n" + "=" * 70)
print("BASIC AGENT (no RAG):")
print("=" * 70)

basic_response = await basic_agent.run(test_query)
print(basic_response.text)

print("\n" + "=" * 70)
print("RAG AGENT (with knowledge base):")
print("=" * 70)

rag_response = await sre_agent.run(test_query, thread=sre_agent.get_new_thread())
print(rag_response.text)

### Key Difference

| Basic Agent | RAG Agent |
|-------------|------------|
| Generic advice from training | Specific steps from YOUR runbooks |
| "Check CPU usage" | "Run `sp_who2` to identify blocking queries" |
| No citations | "According to the Database Performance Runbook..." |
| May hallucinate | Grounded in actual documentation |

---

## Step 8: Explore the Knowledge Base

Try different queries to see semantic search in action.

In [ ]:
# Try different queries - semantic search finds related content!
test_queries = [
    "production is down",           # Should find SEV1 playbook
    "SQL queries are slow",         # Should find query optimization guide
    "incident response process",    # Should find SEV1 playbook
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 40)
    results = search_knowledge_base(query)
    for doc in results[:2]:  # Top 2 results
        print(f"  → {doc['title']} (similarity: {doc['similarity']})")

---

## Summary: What You Learned

| Concept | Key Insight |
|---------|-------------|
| **Vector Store** | Store text as numbers for similarity search |
| **Embedding** | AI turns "slow database" → `[0.23, 0.87, ...]` |
| **RAG** | Agent retrieves docs BEFORE answering → grounded responses |
| **Cosmos DB** | Serverless + DiskANN = cheap, fast vector search |

### One-liner

> **RAG makes your agent use YOUR documentation instead of making things up.**

---

## Cleanup

When you're done with the lab, you can delete the Azure resources:

```bash
az group delete --name rg-agent-labs --yes --no-wait
```

---

## Next Steps

- **Lab 2:** Build an SRE agent that combines metrics + knowledge base for incident response
- **Lab 3:** Add observability and middleware to production-ready agents

---

## References

| Resource | URL |
|----------|-----|
| Cosmos DB Vector Search | https://learn.microsoft.com/azure/cosmos-db/nosql/vector-search |
| Agent Framework Tools | https://learn.microsoft.com/agent-framework/user-guide/agents/agent-tools |
| RAG Pattern | https://learn.microsoft.com/azure/ai-services/openai/concepts/retrieval-augmented-generation |